# FinReasoningAI Colab
End-to-end FinCoT QLoRA training for `Qwen/Qwen2.5-14B-Instruct`, with a standalone PEFT adapter saved for vLLM LoRA serving.

## Step 0a: GPU Check
Expected runtime: under 1 minute. VRAM guidance: training is designed for an A100 with at least 35 GB free VRAM before model load.

In [ ]:
import subprocess

def gpu_summary():
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.free",
            "--format=csv,noheader,nounits",
        ], text=True)
        print(out.strip())
        first = out.strip().splitlines()[0].split(",")
        free_gb = float(first[2].strip()) / 1024.0
        if free_gb < 35:
            print(f"Warning: only {free_gb:.1f} GB free VRAM detected; training may OOM.")
    except Exception as exc:
        print(f"Unable to query GPU details: {exc}")

gpu_summary()

## Step 0b: Mount Google Drive and Infer Workspace
Expected runtime: 1-2 minutes. This cell mounts Drive and auto-detects the notebook workspace instead of hardcoding paths.

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

def _infer_notebook_workspace() -> str:
    roots = [Path("/content/drive/MyDrive"), Path("/content/drive/Shared drives")]
    for root in roots:
        if not root.exists():
            continue
        for candidate in root.rglob("FinReasoningAI_Colab.ipynb"):
            return str(candidate.parent)
    return str(Path("/content/drive/MyDrive"))

DRIVE_BASE = _infer_notebook_workspace()
print("DRIVE_BASE:", DRIVE_BASE)

## Step 0c: Install Missing Dependencies
Expected runtime: 3-8 minutes on a fresh runtime. PyTorch is intentionally not installed here because Colab already provides the GPU build.

In [ ]:
import importlib
import pkg_resources
import subprocess
import sys

REQUIRED = {
    "transformers":  "4.41.0",
    "datasets":      "2.19.0",
    "accelerate":    "0.30.0",
    "peft":          "0.10.0",
    "trl":           "0.8.6",
    "bitsandbytes":  "0.46.1",
    "evaluate":      "0.4.1",
    "rouge-score":   "0.1.2",
    "scikit-learn":  "1.4.0",
    "pandas":        "2.2.0",
    "pydantic":      "2.7.0",
    "jsonlines":     "4.0.0",
}

missing = []
for package, version in REQUIRED.items():
    try:
        installed = pkg_resources.get_distribution(package).version
        if installed != version:
            missing.append(f"{package}=={version}")
    except Exception:
        missing.append(f"{package}=={version}")

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "bitsandbytes"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Restart the runtime before training if packages were freshly installed, especially after upgrading bitsandbytes.")
else:
    print("All required packages already match the requested versions.")

## Step 0d: Clone or Pull the Repository
Expected runtime: under 2 minutes. This cell always refreshes the repo in the current Colab workspace before imports.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/juankim834/FinReasoningAI.git"
workspace = Path(DRIVE_BASE)
project_candidate = workspace / "FinReasoningAI"
PROJECT_DIR = project_candidate if project_candidate.exists() else workspace

if (PROJECT_DIR / ".git").exists():
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"])
else:
    if project_candidate.exists() and any(project_candidate.iterdir()):
        PROJECT_DIR = project_candidate
    else:
        subprocess.check_call(["git", "clone", REPO_URL, str(project_candidate)])
        PROJECT_DIR = project_candidate

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("PROJECT_DIR:", PROJECT_DIR)

## Step 1: Configuration
Expected runtime: immediate. Adjust the knobs here before running the data, training, and evaluation cells below.

In [ ]:
# MODEL
MODEL_ID          = "Qwen/Qwen2.5-14B-Instruct"

# DATA
DATASET_SOURCE    = "huggingface"   # "huggingface" | "local"
LOCAL_DATA_PATH   = ""              # path to local JSONL if DATASET_SOURCE="local"
MAX_SAMPLES       = None            # None = full dataset
INCLUDE_COT       = True
TRAIN_FRAC        = 0.90
VAL_FRAC          = 0.05

# TRAINING
NUM_EPOCHS        = 1
BATCH_SIZE        = 4
GRAD_ACCUM        = 8
LEARNING_RATE     = 2e-4
MAX_SEQ_LEN       = 2048
USE_WANDB         = False

# LoRA
LORA_R            = 64
LORA_ALPHA        = 128
LORA_DROPOUT      = 0.05
LORA_TARGET_MODS  = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# OUTPUT
OUTPUT_DIR        = "outputs/sft_qlora"
ADAPTER_SAVE_DIR  = f"{OUTPUT_DIR}/final_adapter"
EVAL_MAX_SAMPLES  = 200
SKIP_DPO          = True

## Step 2a: Load the Base Model
Expected runtime: 5-10 minutes. VRAM usage after 4-bit load is typically around 10-15 GB before LoRA adapters and activations.

In [ ]:
from src.model.load_model import DEFAULT_BNB_CONFIG, load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    model_id=MODEL_ID,
    bnb_config=DEFAULT_BNB_CONFIG,
)

import torch
if torch.cuda.is_available():
    print(f"Allocated VRAM after base load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 2b: Apply QLoRA
Expected runtime: under 2 minutes. This attaches trainable adapters while keeping the base model quantized for A100-friendly fine-tuning.

In [ ]:
from src.model.apply_lora import apply_qlora
from src.train.sft_train import build_lora_config

lora_config = build_lora_config(
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODS,
)
model = apply_qlora(model, lora_config=lora_config, gradient_checkpointing=True)
model.print_trainable_parameters()

## Step 3a: Load FinCoT Samples
Expected runtime: 1-5 minutes depending on whether the dataset is pulled from Hugging Face or read locally.

In [ ]:
from src.data.fincot_loader import load_fincot_samples

load_source = DATASET_SOURCE
local_path = LOCAL_DATA_PATH or "data/raw/fincot.jsonl"
samples = load_fincot_samples(
    source=load_source,
    local_path=local_path,
    max_samples=MAX_SAMPLES,
)
assert len(samples) > 0
assert all(key in samples[0] for key in ["question", "answer", "chain_of_thought", "task"])

## Step 3b: Inspect One Sample per Task
Expected runtime: under 1 minute. This helps confirm the loader is producing the normalized schema we expect before formatting.

In [ ]:
from collections import defaultdict

examples = defaultdict(list)
for sample in samples:
    examples[sample["task"]].append(sample)

for task, rows in examples.items():
    print("=" * 80)
    print(task)
    print(rows[0])

## Step 3c: Build Prompt/Completion Datasets
Expected runtime: 2-5 minutes. The preprocessing path uses `tokenizer.apply_chat_template` so the chat format stays aligned with Qwen tokenizer updates.

In [ ]:
from src.data.preprocess import build_dataset, format_sample_as_chat
from tools.financial_tools import FINANCIAL_TOOLS

dataset = build_dataset(
    samples,
    tokenizer,
    output_dir="data/processed",
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    include_cot=INCLUDE_COT,
    tool_definitions=FINANCIAL_TOOLS,
)
print({split: len(ds) for split, ds in dataset.items()})
example = format_sample_as_chat(samples[0], tokenizer, include_cot=INCLUDE_COT, tool_definitions=FINANCIAL_TOOLS)
print(example["prompt"][:1500])
print("\n--- COMPLETION ---\n")
print(example["completion"])
assert example["prompt"]
assert example["completion"].startswith("Answer:") or example["completion"].splitlines()[0]

## Step 4: SFT Training (1 Epoch)
Expected runtime: several hours on an A100 depending on dataset size. Peak VRAM will usually sit in the 30-40 GB range with the default sequence length and effective batch size.

In [ ]:
from src.train.sft_train import main as sft_main, save_adapter_for_vllm

trainer = sft_main(
    model_id=MODEL_ID,
    output_dir=OUTPUT_DIR,
    data_dir="data/processed",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_seq_length=MAX_SEQ_LEN,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODS,
    use_wandb=USE_WANDB,
)
save_adapter_for_vllm(trainer=trainer, output_dir=ADAPTER_SAVE_DIR)

## Step 5: Evaluation
Expected runtime: 10-30 minutes for 200 samples depending on decoding mode. This reuses the raw held-out test slice so answer/context fields are available for metrics.

In [ ]:
from importlib import import_module
from pathlib import Path

from src.data.preprocess import load_eval_test_samples
from src.eval.evaluate import evaluate_model
from src.model.load_model import load_model_and_tokenizer

base_model, eval_tokenizer = load_model_and_tokenizer(MODEL_ID)
PeftModel = import_module("peft").PeftModel
eval_model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_DIR)
test_samples = load_eval_test_samples(local_path if DATASET_SOURCE == "local" else "data/raw/fincot.jsonl", train_frac=TRAIN_FRAC, val_frac=VAL_FRAC)
metrics = evaluate_model(
    eval_model,
    eval_tokenizer,
    test_samples,
    output_csv="outputs/eval_results.csv",
    max_samples=EVAL_MAX_SAMPLES,
)
print(metrics)

## Step 6a: Direct Inference Demo
Expected runtime: under 1 minute for a single prompt. This is the shortest path for a basic financial QA response.

In [ ]:
from src.inference.generate import build_prompt, generate_answer

question = "What does a lower debt-to-equity ratio generally suggest about a company's balance sheet?"
prompt = build_prompt(question=question, tokenizer=tokenizer)
print(prompt[:1000])
print()
print(generate_answer(model, tokenizer, question=question, max_new_tokens=128, grounding_check=False))

## Step 6b: Chain-of-Thought Inference Demo
Expected runtime: under 1 minute. This uses the reasoning-oriented inference path and expects the answer to end with an `Answer:` line.

In [ ]:
cot_question = "If revenue grows from 120 to 150, what is the percentage growth?"
print(generate_answer(model, tokenizer, question=cot_question, use_cot=True, max_new_tokens=256, grounding_check=False))

## Step 6c: Tool-Augmented Inference Demo
Expected runtime: under 1 minute. This exercises the tool loop so we can confirm a ratio question either triggers a tool call or still returns a plain answer without crashing.

In [ ]:
from src.inference.generate import generate_with_tools
from tools.financial_tools import FINANCIAL_TOOLS

tool_demo = generate_with_tools(
    model,
    tokenizer,
    question="What is Apple's P/E ratio given net income of 97 and market cap of 2800?",
    tools=FINANCIAL_TOOLS,
    max_new_tokens=256,
)
print(tool_demo)

## Step 6d: Self-Consistency Demo
Expected runtime: a few minutes for 5 samples because each prompt is sampled multiple times. Use this to inspect agreement-based robustness after training.

In [ ]:
from src.inference.self_consistency import sample_with_self_consistency

prompt = build_prompt(question="What is the CAGR from 100 to 121 over 2 periods?", tokenizer=tokenizer, use_cot=True)
final_answer, confidence, raw_answers = sample_with_self_consistency(
    model, tokenizer, prompt, n=5, temperature=0.7, max_new_tokens=128
)
print("Final:", final_answer)
print("Confidence:", confidence)
print(raw_answers)

## Step 7: Optional DPO
Expected runtime: only relevant if you later choose to extend the pipeline beyond the 1-epoch SFT run. This stays disabled by default.

In [ ]:
if SKIP_DPO:
    print("Skipping DPO as configured.")
else:
    print("Add your optional DPO workflow here.")

## Verification Checklist
Expected runtime: under 1 minute after earlier cells complete. These assertions cover the minimum smoke tests requested in the rewrite spec.

In [ ]:
from pathlib import Path

assert len(samples) > 0
formatted = format_sample_as_chat(samples[0], tokenizer, include_cot=INCLUDE_COT)
assert formatted["prompt"] and formatted["completion"]
assert set(dataset.keys()) == {"train", "val", "test"}
assert sum(len(ds) for ds in dataset.values()) == len(samples)
assert Path(ADAPTER_SAVE_DIR).exists()
assert Path(ADAPTER_SAVE_DIR, "adapter_config.json").exists()
assert isinstance(tool_demo["tool_calls"], list)
print("Notebook smoke checks passed.")